# Phase 3 reranker delta bundle
Run with Internet enabled. The output contains only the three new rerankers and the offline runtime bundle.

In [ ]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/AIVIETNAM-AIO-DinhBao/UIT-LegalIR.git'
REPO_REF = 'main'
EXPERIMENT_ID = 'phase3-rerankers-harrier-retrieval'
BUNDLE_ROOT = Path('/kaggle/working/legalir-phase3-reranker-delta')
REPO_DIR = Path('/kaggle/working/UIT-LegalIR-phase3')
def run(*command, cwd=None):
    print('+', ' '.join(map(str, command)))
    subprocess.run(list(map(str, command)), cwd=cwd, check=True)
if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
run('git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, REPO_DIR)
if BUNDLE_ROOT.exists(): shutil.rmtree(BUNDLE_ROOT)
for directory in ('models', 'wheels', 'configs', 'manifests', 'project', 'licenses'): (BUNDLE_ROOT / directory).mkdir(parents=True, exist_ok=True)
print('Bundle output:', BUNDLE_ROOT)

In [ ]:
run(sys.executable, '-m', 'pip', 'download', '--no-deps', '--dest', BUNDLE_ROOT / 'wheels', '--only-binary=:all:', '-r', REPO_DIR / 'requirements-offline.txt')
run(sys.executable, '-m', 'pip', 'wheel', '--no-deps', '--wheel-dir', BUNDLE_ROOT / 'wheels', REPO_DIR)
shutil.copy2(REPO_DIR / 'requirements-offline.txt', BUNDLE_ROOT / 'requirements-offline.txt')
shutil.copy2(REPO_DIR / 'kaggle' / 'phase3_reranker' / 'kaggle_rtx_pro_6000.yaml', BUNDLE_ROOT / 'configs' / 'kaggle_rtx_pro_6000.yaml')
print('Runtime wheelhouse ready')

In [ ]:
from huggingface_hub import snapshot_download
MODELS = {
  'legal_reranker': {'id': 'kiencnt2205/vietnamese-legal-reranker-bge-base', 'revision': 'e6894c39b03a2576972045c4fd3b9a2358eadb72', 'license': 'not-recorded'},
  'qwen3_reranker': {'id': 'Qwen/Qwen3-Reranker-0.6B', 'revision': 'e61197ed45024b0ed8a2d74b80b4d909f1255473', 'license': 'Apache-2.0'},
  'prism_reranker': {'id': 'infgrad/Prism-Qwen3.5-Reranker-0.8B', 'revision': 'c60729783cb2f0662c0b054a02964402a211e3f4', 'license': 'MIT'},
}
for name, spec in MODELS.items():
    destination = BUNDLE_ROOT / 'models' / name
    snapshot_download(repo_id=spec['id'], repo_type='model', revision=spec['revision'], local_dir=destination, ignore_patterns=['onnx/*', '*.onnx', '*.onnx_data'])
    if not (destination / 'config.json').is_file(): raise RuntimeError(f'Missing config for {name}')
    for candidate in ('LICENSE', 'LICENSE.md', 'LICENSE.txt'):
        source = destination / candidate
        if source.is_file(): shutil.copy2(source, BUNDLE_ROOT / 'licenses' / f'{name}_{candidate}')
print('Downloaded:', ', '.join(MODELS))

In [ ]:
import hashlib
def sha256(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''): h.update(block)
    return h.hexdigest()
def records(root):
    return [{'path': str(p.relative_to(BUNDLE_ROOT)), 'bytes': p.stat().st_size, 'sha256': sha256(p)} for p in sorted(root.rglob('*')) if p.is_file()]
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
manifest = {'schema_version': 3, 'experiment_id': EXPERIMENT_ID, 'project_commit': commit, 'models': [{**spec, 'name': name, 'local_path': f'models/{name}'} for name, spec in MODELS.items()], 'files': records(BUNDLE_ROOT / 'models') + records(BUNDLE_ROOT / 'wheels') + records(BUNDLE_ROOT / 'configs')}
(BUNDLE_ROOT / 'manifests' / 'bundle_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
(BUNDLE_ROOT / 'manifests' / 'checksums.sha256').write_text(''.join(f"{x['sha256']}  {x['path']}\n" for x in manifest['files']), encoding='utf-8')
(BUNDLE_ROOT / 'project' / 'source_commit.txt').write_text(commit + '\n', encoding='utf-8')
required = ('faiss_cpu-', 'sentence_transformers-', 'transformers-', 'tokenizers-', 'safetensors-', 'uit_legalir-')
names = [p.name.lower() for p in (BUNDLE_ROOT / 'wheels').glob('*.whl')]
missing = [prefix for prefix in required if not any(n.startswith(prefix) for n in names)]
if missing: raise RuntimeError(f'Missing wheels: {missing}')
print(json.dumps({'experiment_id': EXPERIMENT_ID, 'project_commit': commit, 'files': len(manifest['files'])}, indent=2))